# 23. Logging and Debugging Agents

**Logging** means **printing what the agents are doing**, step by step.

**Debugging** means using those logs to **find and fix problems** when an agent does something wrong.

## Real-life analogy

Without logs, an agent is a **black box** — you only see the final answer.

Logs are like the agent **thinking out loud**, so you can see *why* it did what it did.

## Trick 1: Turn on `verbose=True`

This is the easiest way to watch agents work. Set it on the **agent** and the **crew**.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from crewai import Agent, Task, Crew, LLM

llm = LLM(model="gpt-4o-mini")

agent = Agent(
    role="Helper",
    goal="Answer simply",
    backstory="You help beginners.",
    llm=llm,
    verbose=True,    # <-- print this agent's thinking
)

task = Task(description="Say one fun fact about bees.", expected_output="One fun fact.", agent=agent)

crew = Crew(
    agents=[agent],
    tasks=[task],
    verbose=True,
    tracing = True,    # <-- print the crew's steps too
)

# In a Jupyter notebook, use kickoff_async() with "await".
result = await crew.kickoff_async()
print("\nFINAL:", result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.7                                                                                        │
│  Latest version:  1.15.1                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 31ab3dc9-4d41-49fd-9382-492bf00179cf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Say one fun fact about bees.                                                                             │
│  ID: 640ac034-8cc1-4cbb-b293-276777c8cb18                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Helper                                                                                                  │
│                                                                                                                 │
│  Task: Say one fun fact about bees.                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Helper                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  One fun fact about bees is that they can recognize human faces! They use their excellent memory to remember    │
│  and identify people.                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Say one fun fact about bees.                                                                             │
│  Agent: Helper                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 31ab3dc9-4d41-49fd-9382-492bf00179cf                                                                       │
│  Final Output: One fun fact about bees is that they can recognize human faces! They use their excellent memory  │
│  to remember and identify people.                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL: One fun fact about bees is that they can recognize human faces! They use their excellent memory to remember and identify people.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Trick 2: Read the output object

The result has useful details you can inspect, like the final text and which task produced it.

In [2]:
# 'result' came from crew.kickoff() above
print("Final answer text:")
print(result.raw)

Final answer text:
One fun fact about bees is that they can recognize human faces! They use their excellent memory to remember and identify people.


## Common problems and easy fixes

| Problem | Likely cause | Fix |
|---------|--------------|-----|
| `AuthenticationError` | No / wrong API key | Check your `.env` key |
| Agent ignores a tool | Unclear tool docstring | Write a clearer description |
| Wrong / vague answer | Vague goal or task | Make role, goal, task clearer |
| Agent loops forever | Task too hard | Split into smaller tasks |

## Simple debugging steps

1. Turn on **`verbose=True`** and read the steps.
2. Find the **first** step that looks wrong.
3. Make that agent's **role/goal/task clearer**.
4. Run again and compare.

Small, clear instructions fix most agent problems.

## Key points to remember

- **Logging** = printing what agents do; **debugging** = using that to fix issues.
- The easiest tool is **`verbose=True`** on the agent and crew.
- Inspect `result.raw` for the final text.
- Most problems come from **unclear goals/tasks** or a **missing API key**.
- Fix the **first** wrong step, then re-run.